# 🩺 10-Year Coronary Heart Disease (CHD) Prediction

## 1. Introduction

This Jupyter notebook presents a comprehensive machine learning workflow for predicting the 10-year risk of Coronary Heart Disease (CHD) based on various health and lifestyle factors. CHD is a significant health concern, and early prediction can enable timely interventions and preventative measures.

The dataset contains a range of features, including demographic information, medical history, and physiological measurements. Our goal is to build a robust classification model that can accurately identify individuals at high risk of developing CHD within the next decade.

This notebook will cover:
1.  **Data Loading**: Importing the dataset into a pandas DataFrame.
2.  **Exploratory Data Analysis (EDA)**: Understanding the data distribution, identifying patterns, and checking for missing values and outliers.
3.  **Preprocessing**: Handling missing values, managing outliers, and scaling features.
4.  **Visualizations**: Creating interactive plots to explain data relationships, correlation, and covariance.
5.  **Feature Selection**: Identifying the most relevant features for modeling.
6.  **Model Training**: Applying various classification algorithms.
7.  **Model Evaluation**: Assessing model performance using appropriate metrics.
8.  **Advanced Concepts**: Explaining gradient descent, residuals, and overfitting/underfitting.
9.  **Hyperparameter Tuning**: Optimizing model parameters for better performance.
10. **Final Model Deployment**: Saving the best-performing model.
11. **Insights and Conclusion**: Summarizing findings and future directions.

### Architecture Diagram

```mermaid
graph TD
    A[Raw Data CSV] --> B(Data Loading);
    B --> C{Data Preprocessing};
    C --> D[Missing Value Imputation];
    D --> E[Outlier Handling];
    E --> F[Feature Scaling];
    F --> G[Feature Engineering (Optional)];
    G --> H{Exploratory Data Analysis (EDA)};
    H --> I[Visualizations (Plotly)];
    H --> J[Correlation & Covariance Analysis];
    J --> K[Feature Selection];
    K --> L(Split Data: Train/Test);
    L --> M{Model Training};
    M --> N[Logistic Regression];
    M --> O[Random Forest Classifier];
    M --> P[Gradient Boosting Classifier];
    M --> Q[Support Vector Machine];
    N --> R[Hyperparameter Tuning];
    O --> R;
    P --> R;
    Q --> R;
    R --> S(Model Evaluation);
    S --> T[Accuracy, Precision, Recall, F1-Score, ROC AUC];
    S --> U[Confusion Matrix];
    S --> V[ROC Curve Plot];
    U --> W(Model Comparison & Selection);
    W --> X[Final Model];
    X --> Y[Make Predictions on New Data];
    X --> Z(Save Model to `artifacts/`);
    Z --> AA(Deploy Model);

    subgraph Monitoring & Logging
        C --- MLL(ML Logs);
        S --- MLL;
        Y --- MLL;
        Z --- MLL;
    end
```

*(Note: The actual SVG file should be generated using diagrams.net and placed in the notebook's directory for proper embedding.)*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import logging
import pickle

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE # For handling imbalanced data

# --- Configuration for Logging ---
LOG_DIR = 'ml_logs'
os.makedirs(LOG_DIR, exist_ok=True) # Create directory if it doesn't exist
LOG_FILE = os.path.join(LOG_DIR, 'chd_prediction.log')

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[
                        logging.FileHandler(LOG_FILE),
                        logging.StreamHandler() # Also print to console
                    ])

logging.info(f"Logging configured. Log file: {LOG_FILE}")

# --- Create artifacts directory ---
ARTIFACTS_DIR = 'artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
logging.info(f"Artifacts directory created: {ARTIFACTS_DIR}")

# Set a random seed for reproducibility
np.random.seed(42)


## 2. Data Loading

In this section, we load the dataset from a specified CSV file. We assume the dataset `framingham.csv` is located in a `data` directory relative to this notebook. Robust error handling is implemented to manage potential `FileNotFoundError`.


In [ ]:
DATA_PATH = os.path.join('data', 'framingham.csv') # Assuming data folder in root
df = None
try:
    df = pd.read_csv(DATA_PATH)
    logging.info(f"Dataset successfully loaded from {DATA_PATH}. Shape: {df.shape}")
except FileNotFoundError:
    logging.error(f"Error: The file {DATA_PATH} was not found. Please ensure the CSV file is in the 'data' directory.")
    # Exit or handle gracefully if data loading is critical
    exit()
except Exception as e:
    logging.error(f"An unexpected error occurred during data loading: {e}")
    exit()

if df is not None:
    logging.info("Displaying first 5 rows of the dataset:")
    print(df.head())
    logging.info("Displaying dataset information:")
    df.info()


## 3. Exploratory Data Analysis (EDA)

EDA is a crucial step to understand the dataset's characteristics, distributions, potential issues like missing values and outliers, and relationships between features and the target variable.

### Initial Data Overview

We'll start by checking the dataset's dimensions, column types, and a summary of descriptive statistics. This provides a quick snapshot of the data's structure and content.


In [ ]:
if df is not None:
    logging.info(f"Dataset shape: {df.shape}")
    logging.info("Column information and data types:")
    df.info()
    logging.info("Descriptive statistics for numerical columns:")
    print(df.describe().T)

    # Check for missing values
    logging.info("Checking for missing values across all columns:")
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Count': missing_values, 'Missing %': missing_percentage})
    missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

    if not missing_df.empty:
        logging.warning("Missing values found in the following columns:")
        print(missing_df)
    else:
        logging.info("No missing values found in the dataset.")

    # Check target variable distribution
    logging.info("Distribution of the target variable 'TenYearCHD':")
    target_distribution = df['TenYearCHD'].value_counts(normalize=True) * 100
    print(target_distribution)

    if target_distribution[0] > 75 or target_distribution[1] > 75: # Arbitrary threshold for imbalance
        logging.warning(f"Target variable 'TenYearCHD' is imbalanced. Class 0: {target_distribution[0]:.2f}%, Class 1: {target_distribution[1]:.2f}%.")
    else:
        logging.info("Target variable 'TenYearCHD' appears to be reasonably balanced.")


## 4. Preprocessing

Data preprocessing involves several steps to clean and prepare the raw data for machine learning models. This includes handling missing values, managing outliers, and feature engineering.

### Handling Missing Values

Based on our EDA, several columns have missing values. We will use appropriate imputation strategies. For numerical features, median imputation is generally robust to outliers and maintains the distribution better than mean imputation.


In [ ]:
if df is not None:
    try:
        # Identify columns with missing values
        missing_cols = df.columns[df.isnull().any()].tolist()
        if missing_cols:
            logging.info(f"Handling missing values in columns: {missing_cols}")
            for col in missing_cols:
                if df[col].dtype in ['float64', 'int64']: # Numerical columns
                    median_val = df[col].median()
                    df[col].fillna(median_val, inplace=True)
                    logging.info(f"Filled missing values in '{col}' with median: {median_val}")
                else: # For other types, mode imputation could be used, but all missing here are float64.
                    mode_val = df[col].mode()[0]
                    df[col].fillna(mode_val, inplace=True)
                    logging.info(f"Filled missing values in '{col}' with mode: {mode_val}")
        else:
            logging.info("No missing values to handle.")

        # Verify no missing values remain
        if df.isnull().sum().sum() == 0:
            logging.info("Missing value imputation complete. Dataset is now free of missing values.")
        else:
            logging.warning("Missing values still detected after imputation. Re-checking...")
            print(df.isnull().sum()[df.isnull().sum() > 0])

    except Exception as e:
        logging.error(f"Error during missing value imputation: {e}")


### Outlier Handling

Outliers can significantly affect model performance. We will detect and handle outliers in numerical features using the Interquartile Range (IQR) method. Outliers detected will be capped at the upper and lower bounds to preserve data points while mitigating their extreme influence.


In [ ]:
if df is not None:
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    # Exclude binary columns and the target variable from outlier treatment if they represent categories
    binary_cols = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'TenYearCHD']
    cols_for_outlier_handling = [col for col in numerical_cols if col not in binary_cols]

    logging.info(f"Applying outlier capping (IQR method) for columns: {cols_for_outlier_handling}")

    try:
        for col in cols_for_outlier_handling:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            # Cap outliers
            initial_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
            if initial_outliers > 0:
                df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
                df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])
                logging.info(f"Capped {initial_outliers} outliers in column '{col}' at [{lower_bound:.2f}, {upper_bound:.2f}].")
            else:
                logging.info(f"No significant outliers found in column '{col}' to cap.")
    except Exception as e:
        logging.error(f"Error during outlier handling: {e}")


### Feature Engineering (Optional)

For this dataset, explicit feature engineering might not be immediately necessary as the existing features are well-defined medical parameters. However, we could consider creating interaction terms or polynomial features if initial models underperform. For now, we will proceed with the existing features.

One potential, simple engineered feature could be 'Age_BMI_Interaction' or 'BP_category', but let's stick to the base features first to establish a baseline.


In [ ]:
if df is not None:
    logging.info("Feature Engineering: No new features engineered at this stage.")
    # Example for future consideration:
    # df['Age_x_BMI'] = df['age'] * df['BMI']
    # logging.info("Created 'Age_x_BMI' interaction feature.")


## 5. Visual Representation of EDA

Visualizations are essential to gain deeper insights into the data. We will use the `plotly` library to create interactive and informative plots, covering distributions of individual features and their relationships with the target variable.

### Target Variable Distribution

Understanding the balance of our target variable (`TenYearCHD`) is crucial, especially for classification tasks. An imbalanced target can lead to models biased towards the majority class.


In [ ]:
if df is not None:
    try:
        fig = px.pie(df, names='TenYearCHD', title='Distribution of TenYearCHD (10-year risk of Coronary Heart Disease)',
                     hole=0.3, color_discrete_sequence=px.colors.qualitative.Pastel)
        fig.update_traces(textinfo='percent+label', marker=dict(line=dict(color='#000000', width=1)))
        fig.show()
        logging.info("Displayed pie chart for TenYearCHD distribution.")
    except Exception as e:
        logging.error(f"Error plotting TenYearCHD distribution: {e}")


The pie chart above clearly shows the distribution of our target variable, `TenYearCHD`. We can observe that a significantly larger proportion of individuals (class 0) do not develop CHD within 10 years, compared to those who do (class 1). This indicates a class imbalance, which we will need to address during model training to prevent the model from becoming biased towards the majority class.

### Numerical Feature Distributions

Histograms and KDE plots help visualize the distribution of continuous numerical features. This can reveal skewness, multi-modality, and potential data quality issues.


In [ ]:
if df is not None:
    numerical_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
    fig = make_subplots(rows=len(numerical_cols)//2 + len(numerical_cols)%2, cols=2,
                        subplot_titles=[f'Distribution of {col}' for col in numerical_cols])

    for i, col in enumerate(numerical_cols):
        row = (i // 2) + 1
        col_idx = (i % 2) + 1
        fig.add_trace(go.Histogram(x=df[col], name=col, marker_color=px.colors.qualitative.Plotly[i%10]),
                      row=row, col=col_idx)
        fig.update_xaxes(title_text=col, row=row, col=col_idx)
        fig.update_yaxes(title_text='Count', row=row, col=col_idx)

    fig.update_layout(height=400 * (len(numerical_cols)//2 + len(numerical_cols)%2), showlegend=False,
                      title_text="Histograms of Numerical Features")
    fig.show()
    logging.info("Displayed histograms for numerical features.")


These histograms illustrate the distributions of our key numerical features.
*   `age`: Shows a relatively uniform distribution across the middle-aged population, indicating a good spread of ages in the dataset.
*   `cigsPerDay`: Heavily skewed towards 0, as many individuals are non-smokers, which is expected.
*   `totChol`, `sysBP`, `diaBP`, `BMI`, `heartRate`, `glucose`: Generally show distributions that are somewhat normally distributed or slightly skewed, which is common for biological measurements. Some features like `sysBP` and `diaBP` might show a tendency towards higher values due to the prevalence of hypertension in the population, which is a risk factor for CHD.

### Relationship of Features with TenYearCHD

Box plots are excellent for visualizing the relationship between a numerical feature and a categorical target variable.


In [ ]:
if df is not None:
    fig = make_subplots(rows=len(numerical_cols)//2 + len(numerical_cols)%2, cols=2,
                        subplot_titles=[f'{col} vs. TenYearCHD' for col in numerical_cols])

    for i, col in enumerate(numerical_cols):
        row = (i // 2) + 1
        col_idx = (i % 2) + 1
        fig.add_trace(go.Box(x=df['TenYearCHD'], y=df[col], name=col, marker_color=px.colors.qualitative.Plotly[i%10]),
                      row=row, col=col_idx)
        fig.update_xaxes(title_text='TenYearCHD', row=row, col=col_idx)
        fig.update_yaxes(title_text=col, row=row, col=col_idx)

    fig.update_layout(height=400 * (len(numerical_cols)//2 + len(numerical_cols)%2), showlegend=False,
                      title_text="Box Plots of Numerical Features by TenYearCHD")
    fig.show()
    logging.info("Displayed box plots comparing numerical features with TenYearCHD.")


The box plots illustrate how the distribution of each numerical feature varies between individuals who develop CHD (TenYearCHD=1) and those who don't (TenYearCHD=0).
*   **`age`**: Older individuals tend to have a higher risk of CHD, as indicated by a higher median age for `TenYearCHD=1`.
*   **`sysBP`, `diaBP`**: Higher systolic and diastolic blood pressure values are associated with a greater likelihood of CHD.
*   **`totChol`, `glucose`**: Individuals with higher total cholesterol and glucose levels also show a higher median for `TenYearCHD=1`.
*   **`cigsPerDay`, `BMI`, `heartRate`**: While showing some overlap, there's a general trend of slightly higher values for `TenYearCHD=1` in these features, suggesting they are also risk factors.

These visualizations confirm that many of the features are indeed relevant to the prediction of CHD.

### Categorical Feature Distributions

For binary/categorical features, bar plots show the counts or proportions within each category.


In [ ]:
if df is not None:
    binary_cols = ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']
    fig = make_subplots(rows=len(binary_cols)//2 + len(binary_cols)%2, cols=2,
                        subplot_titles=[f'Distribution of {col}' for col in binary_cols])

    for i, col in enumerate(binary_cols):
        row = (i // 2) + 1
        col_idx = (i % 2) + 1
        counts = df[col].value_counts().reset_index()
        counts.columns = [col, 'count']
        fig.add_trace(go.Bar(x=counts[col].astype(str), y=counts['count'], name=col,
                             marker_color=px.colors.qualitative.Set3[i%10]),
                      row=row, col=col_idx)
        fig.update_xaxes(title_text=col, row=row, col=col_idx)
        fig.update_yaxes(title_text='Count', row=row, col=col_idx)

    fig.update_layout(height=400 * (len(binary_cols)//2 + len(binary_cols)%2), showlegend=False,
                      title_text="Bar Plots of Binary/Categorical Features")
    fig.show()
    logging.info("Displayed bar plots for binary/categorical features.")


These bar plots show the distribution of our binary/categorical features:
*   **`male`**: The dataset contains a relatively balanced number of males and females.
*   **`education`**: Shows the distribution of education levels, with certain levels being more common.
*   **`currentSmoker`**: A significant portion of the population are non-smokers.
*   **`BPMeds`, `prevalentStroke`, `prevalentHyp`, `diabetes`**: These show that the majority of the population does not have these conditions, which is expected for prevalent but not universal health issues. The presence of these conditions, even in smaller numbers, makes them important risk factors for CHD.


## 6. Visual Representation of Correlation, Covariance

Understanding the relationships between features is critical. Correlation measures the linear relationship between two variables, while covariance measures how two variables vary together. A heatmap of the correlation matrix is an excellent way to visualize these relationships across all features.


In [ ]:
if df is not None:
    try:
        # Calculate the correlation matrix
        corr_matrix = df.corr()

        # Plotting the correlation heatmap
        fig = px.imshow(corr_matrix,
                        text_auto=True,
                        aspect="auto",
                        color_continuous_scale='RdBu_r',
                        title='Correlation Matrix of Features')
        fig.update_layout(height=800, width=800)
        fig.show()
        logging.info("Displayed correlation matrix heatmap.")

        # Covariance matrix (less interpretable directly, but good to know)
        # For visualization, correlation is usually preferred due to normalization
        # cov_matrix = df.cov()
        # fig_cov = px.imshow(cov_matrix, text_auto=True, aspect="auto", title='Covariance Matrix of Features')
        # fig_cov.update_layout(height=800, width=800)
        # fig_cov.show()
        # logging.info("Displayed covariance matrix heatmap.")

    except Exception as e:
        logging.error(f"Error plotting correlation/covariance: {e}")


### Explaining the Correlation Plot

The heatmap above displays the Pearson correlation coefficients between all pairs of features in our dataset.
*   **Color Scale**: Red shades indicate a positive correlation (as one variable increases, the other tends to increase). Blue shades indicate a negative correlation (as one variable increases, the other tends to decrease). The intensity of the color indicates the strength of the correlation (darker colors mean stronger correlation).
*   **Diagonal**: The diagonal elements are always 1, representing the correlation of a variable with itself.
*   **Symmetry**: The matrix is symmetric; the correlation between Feature A and Feature B is the same as between Feature B and Feature A.

**Key Observations from the Correlation Matrix:**
*   **Target Variable (`TenYearCHD`)**: We are particularly interested in the correlations with `TenYearCHD`.
    *   Strong positive correlations exist with `age`, `sysBP`, `glucose`, `prevalentHyp`, `diabetes`, and `male`. This indicates that older age, higher blood pressure, higher glucose, presence of hypertension, diabetes, and being male are associated with a higher risk of CHD.
    *   `cigsPerDay` also shows a positive correlation, as expected.
*   **Inter-feature Correlations**:
    *   `sysBP` and `diaBP` are highly positively correlated, which is medically expected as they are both measures of blood pressure.
    *   `age` shows positive correlations with `sysBP`, `diaBP`, `totChol`, and `glucose`, suggesting that these physiological measures tend to increase with age.
    *   `currentSmoker` and `cigsPerDay` are highly correlated, which is obvious (if you smoke, you have cigarettes per day).
    *   `prevalentHyp` (prevalent hypertension) is positively correlated with `sysBP` and `diaBP`, as hypertension is defined by high blood pressure.

**Implications:**
*   **Feature Selection**: Highly correlated features (e.g., `sysBP` and `diaBP`) might provide redundant information. While models like Random Forest can handle this, for linear models or to reduce dimensionality, one might choose to keep only one of them or combine them.
*   **Model Interpretation**: Understanding these correlations helps in interpreting which features have the strongest linear relationship with the target, providing insights into CHD risk factors.

**Covariance**: The covariance matrix (not explicitly plotted but conceptually understood) measures the directional relationship between variables, without normalization. Unlike correlation, its values depend on the scales of the variables, making it harder to compare relationships across different pairs directly. For ML, correlation is generally preferred for feature analysis due to its normalized scale (-1 to 1).


## 7. Feature Selection based on EDA

Based on our EDA and correlation analysis, we will select a set of features that are most relevant to predicting `TenYearCHD`. We aim to include features that show a strong relationship with the target variable and avoid highly redundant features where possible.

**Selected Features and Justification:**
*   `age`: Strong positive correlation with CHD risk.
*   `male`: Biological sex is a known risk factor for CHD.
*   `education`: While not strongly correlated, it can be a proxy for socioeconomic status or lifestyle, which might indirectly influence health.
*   `currentSmoker`: Direct risk factor.
*   `cigsPerDay`: Direct measure of smoking intensity, highly correlated with `currentSmoker`, but provides more granularity. We will keep `cigsPerDay` and `currentSmoker` as both have medical relevance, and tree-based models can handle collinearity.
*   `BPMeds`: Indicates current medication for blood pressure, a direct health indicator.
*   `prevalentStroke`: History of stroke is a significant cardiovascular risk.
*   `prevalentHyp`: Indicates a history of hypertension, a major CHD risk factor.
*   `diabetes`: A strong independent risk factor for CHD.
*   `totChol`: Total cholesterol level, a key lipid profile marker for heart health.
*   `sysBP`: Systolic blood pressure, a primary indicator of hypertension and CHD risk.
*   `diaBP`: Diastolic blood pressure, also important. Given its high correlation with `sysBP`, we will keep both as they represent slightly different aspects of blood pressure.
*   `BMI`: Body Mass Index, an indicator of obesity, a known risk factor.
*   `heartRate`: Heart rate, can indicate cardiovascular health status.
*   `glucose`: Glucose levels, directly related to diabetes and metabolic health.

All these features have a demonstrable relationship with `TenYearCHD` based on our EDA and are medically recognized risk factors. We will use all of them as they contribute unique information or provide granularity that models can leverage.


In [ ]:
if df is not None:
    # Define features (X) and target (y)
    # All columns except the target variable 'TenYearCHD' are potential features
    features = [col for col in df.columns if col != 'TenYearCHD']
    target = 'TenYearCHD'

    X = df[features]
    y = df[target]

    logging.info(f"Selected {len(features)} features for modeling: {features}")
    logging.info(f"Target variable: {target}")
    logging.info(f"Shape of X: {X.shape}, Shape of y: {y.shape}")
else:
    logging.error("DataFrame not loaded, cannot proceed with feature selection.")


## 8. Separate the selected features for training

Before training, the dataset must be split into training and testing sets. This allows us to train the model on one subset of data and evaluate its performance on unseen data, which simulates its real-world generalization ability. We will also apply feature scaling to ensure that features with larger numerical ranges do not disproportionately influence the model.

**Why selected features are taken:**
The features selected in the previous step are chosen because they represent a comprehensive set of known medical and lifestyle risk factors for Coronary Heart Disease. EDA showed their individual distributions and relationships with the target variable, indicating their predictive power. Including a diverse set of these clinically relevant features helps the model learn a more nuanced understanding of CHD risk.


In [ ]:
if df is not None:
    try:
        # Split the data into training and testing sets
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
        logging.info(f"Data split into training (X_train: {X_train.shape}, y_train: {y_train.shape}) "
                     f"and testing (X_test: {X_test.shape}, y_test: {y_test.shape}) sets.")
        logging.info("Using 'stratify=y' to maintain the proportion of target classes in both train and test sets, crucial for imbalanced datasets.")

        # Identify numerical features for scaling
        numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()

        # Apply StandardScaler
        scaler = StandardScaler()
        X_train_scaled = X_train.copy()
        X_test_scaled = X_test.copy()

        X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
        X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

        logging.info("Numerical features scaled using StandardScaler.")

        # Handle imbalanced dataset using SMOTE on the training data
        # Check for imbalance first
        if y_train.value_counts(normalize=True)[0] > 0.75 or y_train.value_counts(normalize=True)[1] > 0.75:
            logging.warning("Training target variable is imbalanced. Applying SMOTE to the training data.")
            smote = SMOTE(random_state=42)
            X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
            logging.info(f"SMOTE applied. Original training shape: {X_train_scaled.shape}, Resampled training shape: {X_train_res.shape}")
            logging.info(f"Resampled target distribution: {y_train_res.value_counts(normalize=True)}")
            X_train_final = X_train_res
            y_train_final = y_train_res
        else:
            logging.info("Training target variable is not significantly imbalanced. SMOTE not applied.")
            X_train_final = X_train_scaled
            y_train_final = y_train

    except Exception as e:
        logging.error(f"Error during data splitting or scaling: {e}")
else:
    logging.error("DataFrame not loaded, cannot proceed with data separation.")


## 9. Modeling

This is a binary classification problem, where the goal is to predict `TenYearCHD` (0 or 1). We will select a few appropriate classification models known for their performance and interpretability on tabular data.

**Selected Models and Justification:**
1.  **Logistic Regression**: A simple yet powerful linear model. It provides probabilities and is highly interpretable, serving as a good baseline. It assumes a linear relationship between features and the log-odds of the target.
2.  **Random Forest Classifier**: An ensemble tree-based model. It's robust to overfitting, handles non-linear relationships, and is less sensitive to outliers and scaling. It aggregates predictions from multiple decision trees to improve accuracy and reduce variance.
3.  **Gradient Boosting Classifier (e.g., LightGBM or XGBoost - using scikit-learn's for simplicity)**: Another powerful ensemble method that builds trees sequentially, with each tree trying to correct the errors of the previous ones. Known for high performance.
4.  **Support Vector Machine (SVC)**: A powerful model that finds an optimal hyperplane to separate classes. It can handle non-linear decision boundaries using different kernels, but can be sensitive to feature scaling and computational complexity for large datasets.

We will train these models and prepare for evaluation.


In [ ]:
if 'X_train_final' in locals() and 'y_train_final' in locals():
    models = {
        'Logistic Regression': LogisticRegression(random_state=42, solver='liblinear'), # liblinear for small datasets/binary
        'Random Forest Classifier': RandomForestClassifier(random_state=42, n_estimators=100),
        'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42, n_estimators=100),
        'Support Vector Classifier': SVC(random_state=42, probability=True) # probability=True for ROC AUC
    }

    trained_models = {}
    for name, model in models.items():
        try:
            logging.info(f"Training {name}...")
            model.fit(X_train_final, y_train_final)
            trained_models[name] = model
            logging.info(f"{name} trained successfully.")
        except Exception as e:
            logging.error(f"Error training {name}: {e}")
else:
    logging.error("Training data not prepared. Cannot proceed with modeling.")


## 10. Evaluation metrics suitable for the tasks

For a binary classification task, especially with an imbalanced dataset, common accuracy metrics alone might be misleading. We need a suite of metrics to thoroughly evaluate model performance:

*   **Accuracy**: Proportion of correctly predicted instances out of total instances. (Good for balanced datasets).
*   **Precision**: Proportion of true positive predictions among all positive predictions. (Minimizes False Positives, important when cost of FP is high).
*   **Recall (Sensitivity)**: Proportion of true positive predictions among all actual positive instances. (Minimizes False Negatives, important when cost of FN is high, e.g., missing a CHD case).
*   **F1-Score**: Harmonic mean of precision and recall. Provides a balance between the two, especially useful for imbalanced datasets.
*   **ROC AUC (Receiver Operating Characteristic - Area Under Curve)**: Measures the ability of a classifier to distinguish between classes. A higher AUC indicates better discriminatory power. It is robust to imbalanced datasets.
*   **Confusion Matrix**: A table showing the counts of True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN). This gives a detailed breakdown of correct and incorrect predictions for each class.


In [ ]:
if 'X_test_scaled' in locals() and 'y_test' in locals() and trained_models:
    results = {}
    roc_curves = {}

    for name, model in trained_models.items():
        try:
            y_pred = model.predict(X_test_scaled)
            y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test)

            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)
            roc_auc = roc_auc_score(y_test, y_proba) if hasattr(model, 'predict_proba') else 0

            results[name] = {
                'Accuracy': accuracy,
                'Precision': precision,
                'Recall': recall,
                'F1-Score': f1,
                'ROC AUC': roc_auc
            }
            logging.info(f"Evaluation for {name}:")
            logging.info(f"  Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}, ROC AUC: {roc_auc:.4f}")

            # Store ROC curve data
            if hasattr(model, 'predict_proba'):
                fpr, tpr, _ = roc_curve(y_test, y_proba)
                roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}

            # Confusion Matrix
            cm = confusion_matrix(y_test, y_pred)
            logging.info(f"  Confusion Matrix for {name}:\n{cm}")

        except Exception as e:
            logging.error(f"Error evaluating {name}: {e}")

    # Display results in a DataFrame
    results_df = pd.DataFrame(results).T
    print("\n--- Model Evaluation Results ---")
    print(results_df.sort_values(by='F1-Score', ascending=False))

    # Plot ROC curves
    fig_roc = go.Figure()
    fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier (AUC = 0.50)',
                                 line=dict(dash='dash', color='gray')))

    for name, data in roc_curves.items():
        fig_roc.add_trace(go.Scatter(x=data['fpr'], y=data['tpr'], mode='lines',
                                     name=f'{name} (AUC = {data["auc"]:.2f})'))

    fig_roc.update_layout(title='ROC Curve Comparison',
                          xaxis_title='False Positive Rate',
                          yaxis_title='True Positive Rate',
                          xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
                          showlegend=True)
    fig_roc.show()
    logging.info("Displayed ROC curve comparison for all models.")

else:
    logging.error("Test data or trained models not available for evaluation.")


## 11. Local minima vs Global minima and Visual representation of gradients descent using the dataset.

### Local Minima vs. Global Minima

In the context of machine learning, especially with optimization algorithms like gradient descent, we are often trying to minimize a **cost function** (or loss function).

*   **Global Minimum**: This is the lowest possible value of the cost function across the entire domain of the function. It represents the absolute best set of parameters for our model.
*   **Local Minimum**: This is a point where the cost function is lower than all its neighboring points, but not necessarily the absolute lowest value across the entire domain. An optimization algorithm might get "stuck" here if it doesn't have mechanisms to escape.

**Visual Analogy**: Imagine a mountainous landscape. The global minimum is the lowest valley in the entire range. A local minimum is a small dip or pit in the terrain that is lower than its immediate surroundings, but there are deeper valleys elsewhere.

**Implications for ML**:
*   Convex cost functions (like in simple Linear Regression or Logistic Regression without complex regularizations) have only one global minimum, making optimization straightforward.
*   Non-convex cost functions (common in Neural Networks, some complex SVM kernels, or deep learning) have multiple local minima. Gradient descent might converge to a local minimum, leading to a suboptimal model. Techniques like momentum, adaptive learning rates, or using different initializations can help escape local minima.

### Visual Representation of Gradient Descent

Gradient descent is an iterative optimization algorithm used to find the minimum of a cost function. It works by taking steps proportional to the negative of the gradient (or slope) of the function at the current point. The size of these steps is determined by the **learning rate**.

For a multi-dimensional dataset and complex models, visualizing the entire cost surface and the gradient descent path is difficult. However, we can illustrate the concept using a simplified 1D or 2D example.

Let's illustrate the concept using a simple linear relationship, for instance, trying to predict a simplified `sysBP` based on `age`, and visualize how the error changes with a single weight parameter. (Note: For a classification problem, the cost function is more complex, but the principle remains the same).

We'll use a very simplified quadratic function to represent a cost surface for demonstration, as this clearly shows a minimum and how gradient descent would approach it.


In [ ]:
# Conceptual visualization of Gradient Descent
# We'll use a simple quadratic function to illustrate the concept.
# This is not directly training on our full dataset, but demonstrating the principle.

try:
    # Define a simple quadratic cost function (parabola)
    # J(theta) = (theta - 2)^2 + 1
    # Global minimum at theta = 2, J(theta) = 1
    def cost_function(theta):
        return (theta - 2)**2 + 1

    # Define the derivative of the cost function (gradient)
    # dJ/d(theta) = 2 * (theta - 2)
    def gradient(theta):
        return 2 * (theta - 2)

    # Gradient Descent parameters
    learning_rate = 0.1
    num_iterations = 20
    initial_theta = 0 # Start point

    # Store values for visualization
    theta_history = [initial_theta]
    cost_history = [cost_function(initial_theta)]

    # Perform Gradient Descent
    theta = initial_theta
    for i in range(num_iterations):
        grad = gradient(theta)
        theta = theta - learning_rate * grad
        theta_history.append(theta)
        cost_history.append(cost_function(theta))

    # Create the plot
    theta_vals = np.linspace(-1, 5, 100)
    cost_vals = cost_function(theta_vals)

    fig = go.Figure()

    # Plot the cost function
    fig.add_trace(go.Scatter(x=theta_vals, y=cost_vals, mode='lines', name='Cost Function',
                             line=dict(color='blue')))

    # Plot the gradient descent path
    fig.add_trace(go.Scatter(x=theta_history, y=cost_history, mode='markers+lines', name='GD Path',
                             marker=dict(color='red', size=8), line=dict(color='red', dash='dot')))

    fig.update_layout(title='Gradient Descent Visualization on a Simple Cost Function',
                      xaxis_title='Model Parameter (theta)',
                      yaxis_title='Cost (J(theta))',
                      showlegend=True)
    fig.show()
    logging.info("Displayed conceptual visualization of Gradient Descent.")

except Exception as e:
    logging.error(f"Error during Gradient Descent visualization: {e}")


**Explanation of the Gradient Descent Plot:**
The blue curve represents a hypothetical cost function (a simple parabola in this case). The global minimum for this function is at `theta = 2`, where the cost is `1`.

The red line with markers shows the path taken by the gradient descent algorithm. Starting from an `initial_theta` of 0, the algorithm iteratively moves towards the minimum. In each step, it calculates the gradient (slope) of the cost function at its current position and takes a step in the opposite direction of the gradient.
*   The steps become smaller as the algorithm approaches the minimum, because the gradient becomes less steep.
*   The `learning_rate` controls the size of these steps. A too-large learning rate can cause the algorithm to overshoot the minimum, while a too-small one can make convergence very slow.

This visualization demonstrates how gradient descent converges to the global minimum for a convex function. For complex, non-convex functions, it might converge to a local minimum instead.


## 12. Residuals and how to visualize it. Explain the comparison and the metrics to suggest how to improve them.

### Residuals in Classification

Traditionally, residuals are most directly interpreted in regression tasks, where they represent the difference between the observed and predicted continuous values (actual - predicted). For classification, where the output is a discrete class (0 or 1) or a probability, the concept of "residuals" is adapted.

In classification, we can think of residuals as:
1.  **Probability-based Residuals**: The difference between the actual class (0 or 1) and the predicted probability of belonging to the positive class. For instance, if an individual is `TenYearCHD=1` but the model predicts a probability of 0.2, the "residual" (error) is 0.8.
2.  **Misclassification Analysis**: More commonly, for classification, we analyze which instances were **misclassified** (False Positives and False Negatives) rather than a continuous residual value. This is typically done using a **Confusion Matrix**.

### How to Visualize (Misclassification Analysis for Classification)

A confusion matrix provides a direct visualization of the types of errors a classification model makes.

**Confusion Matrix Visualization:**
A heatmap of the confusion matrix makes it easy to see the counts of TP, TN, FP, and FN.

*   **True Positives (TP)**: Correctly predicted actual positive (correctly identified CHD).
*   **True Negatives (TN)**: Correctly predicted actual negative (correctly identified non-CHD).
*   **False Positives (FP)**: Incorrectly predicted as positive (predicted CHD, but not CHD). Also known as Type I error.
*   **False Negatives (FN)**: Incorrectly predicted as negative (predicted non-CHD, but is CHD). Also known as Type II error.


In [ ]:
if 'X_test_scaled' in locals() and 'y_test' in locals() and trained_models:
    logging.info("Visualizing Confusion Matrices for the models.")

    num_models = len(trained_models)
    rows = (num_models + 1) // 2
    cols = 2 if num_models > 0 else 1 # Ensure at least 1 column for 0 or 1 models

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f'Confusion Matrix: {name}' for name in trained_models.keys()])

    for i, (name, model) in enumerate(trained_models.items()):
        row = (i // cols) + 1
        col = (i % cols) + 1
        try:
            y_pred = model.predict(X_test_scaled)
            cm = confusion_matrix(y_test, y_pred)
            
            # Create a heatmap for the confusion matrix
            z = [[cm[0,0], cm[0,1]], [cm[1,0], cm[1,1]]]
            x = ['Predicted 0', 'Predicted 1']
            y = ['Actual 0', 'Actual 1']
            
            heatmap_trace = go.Heatmap(z=z, x=x, y=y, colorscale='Blues',
                                       text=[[str(val) for val in row_cm] for row_cm in cm],
                                       texttemplate="%{text}", textfont={"size":15})
            fig.add_trace(heatmap_trace, row=row, col=col)

            # Update layout to center titles
            fig.update_layout(height=400 * rows, width=400 * cols)
            fig.update_xaxes(title_text="Predicted Class", row=row, col=col)
            fig.update_yaxes(title_text="Actual Class", row=row, col=col)

        except Exception as e:
            logging.error(f"Error generating confusion matrix for {name}: {e}")

    fig.update_layout(title_text='Confusion Matrices for Different Models')
    fig.show()
    logging.info("Displayed confusion matrices for all models.")

    # A more advanced "residual" analysis could involve looking at the distributions of features for FP/FN
    # Example for one model (e.g., Random Forest)
    if 'Random Forest Classifier' in trained_models:
        rf_model = trained_models['Random Forest Classifier']
        y_pred_rf = rf_model.predict(X_test_scaled)

        # Get misclassified samples
        misclassified_indices = y_test[y_test != y_pred_rf].index
        misclassified_df = df.loc[misclassified_indices]
        misclassified_actual = y_test.loc[misclassified_indices]
        misclassified_predicted = pd.Series(y_pred_rf, index=y_test.index).loc[misclassified_indices]

        logging.info(f"\n--- Analysis of Misclassified Samples (Random Forest) ---")
        logging.info(f"Total misclassified samples: {len(misclassified_indices)}")
        print("Misclassified Samples (Actual vs Predicted):\n", pd.DataFrame({'Actual': misclassified_actual, 'Predicted': misclassified_predicted}).head())

        # Further analysis: distributions of key features for FP vs FN
        # FP: Actual 0, Predicted 1
        fp_indices = y_test[(y_test == 0) & (y_pred_rf == 1)].index
        fp_df = df.loc[fp_indices]
        # FN: Actual 1, Predicted 0
        fn_indices = y_test[(y_test == 1) & (y_pred_rf == 0)].index
        fn_df = df.loc[fn_indices]

        logging.info(f"Number of False Positives (Actual 0, Predicted 1): {len(fp_df)}")
        logging.info(f"Number of False Negatives (Actual 1, Predicted 0): {len(fn_df)}")

        # Example: Compare 'age' distribution for FP and FN
        if not fp_df.empty and not fn_df.empty:
            fig_err_age = go.Figure()
            fig_err_age.add_trace(go.Violin(y=fp_df['age'], name='False Positives (Age)', box_visible=True, meanline_visible=True, fillcolor='lightblue', line_color='blue'))
            fig_err_age.add_trace(go.Violin(y=fn_df['age'], name='False Negatives (Age)', box_visible=True, meanline_visible=True, fillcolor='lightcoral', line_color='red'))
            fig_err_age.update_layout(title='Age Distribution of False Positives vs. False Negatives (Random Forest)',
                                      yaxis_title='Age')
            fig_err_age.show()
            logging.info("Displayed age distribution for FP vs FN.")
        else:
            logging.warning("Not enough false positives or false negatives to plot age distribution.")

else:
    logging.error("Test data or trained models not available for misclassification analysis.")



### Comparison and Metrics to Improve Residuals (Misclassifications)

By analyzing the confusion matrix and misclassified samples, we can identify patterns and suggest improvements:

*   **High False Positives (FP)**: The model predicts CHD when there is none.
    *   **Metrics affected**: Low Precision.
    *   **Improvement suggestions**:
        *   **Increase classification threshold**: If using probabilities, raising the threshold for classifying as '1' will reduce FPs but might increase FNs.
        *   **Feature Engineering**: Are there features that distinguish genuine CHD cases from look-alikes?
        *   **Collect more data**: Specifically, more true negative samples.
        *   **Cost-sensitive learning**: Assign a higher penalty for FPs during training.
        *   **Regularization**: To simplify the model and reduce complex decision boundaries that might lead to FP.

*   **High False Negatives (FN)**: The model predicts no CHD when there is CHD. This is often more critical in medical diagnosis as it means missing a high-risk patient.
    *   **Metrics affected**: Low Recall.
    *   **Improvement suggestions**:
        *   **Decrease classification threshold**: Lowering the threshold for classifying as '1' will increase recall but might also increase FPs.
        *   **Handle class imbalance**: Techniques like SMOTE (as applied), oversampling the minority class, or using `class_weight` in models can help the model pay more attention to the positive class.
        *   **Feature Engineering**: Are there subtle features that the model missed to identify actual CHD cases?
        *   **Collect more data**: Especially true positive samples.
        *   **Ensemble methods**: More complex models like Random Forest or Gradient Boosting often perform better on minority classes.

By visualizing the features of FP vs. FN (e.g., age distribution above), we might find that false negatives are concentrated among younger individuals with slightly lower risk factors, while false positives might occur in older individuals who present with some risk factors but aren't actually CHD cases. This deeper dive helps in refining feature sets or re-balancing the data more effectively.


## 13. Overfitting or Underfitting if it exists. Explain how to fix it.

Understanding overfitting and underfitting is crucial for building effective machine learning models.

*   **Overfitting**: This occurs when a model learns the training data too well, including its noise and specific patterns, making it perform poorly on unseen (test) data. The model is too complex for the training data.
    *   **Signs**: High accuracy/performance on the training set, but significantly lower accuracy/performance on the test set. The model fails to generalize.
    *   **Analogy**: A student who memorizes answers for a specific exam but doesn't understand the underlying concepts will perform poorly on a new, slightly different exam.

*   **Underfitting**: This occurs when a model is too simple to capture the underlying patterns in the training data, leading to poor performance on both training and test data. The model has not learned enough from the data.
    *   **Signs**: Low accuracy/performance on both the training and test sets.
    *   **Analogy**: A student who doesn't study enough and therefore performs poorly on any exam.

### How to Fix Overfitting:
1.  **More Data**: The most effective solution. A larger and more diverse dataset helps the model learn generalized patterns rather than specific noise.
2.  **Feature Reduction/Selection**: Remove irrelevant or redundant features that might be contributing to noise.
3.  **Regularization**: Add a penalty to the cost function for complex models (e.g., L1/L2 regularization in Logistic Regression, Lasso/Ridge Regression). This discourages large coefficients.
4.  **Simpler Models**: Choose a less complex model with fewer parameters.
5.  **Cross-validation**: Helps to get a more reliable estimate of model performance and detect overfitting early.
6.  **Early Stopping**: For iterative models (like Gradient Boosting), stop training when validation performance starts to degrade.
7.  **Ensemble Methods**: Bagging (e.g., Random Forest) reduces variance and helps prevent overfitting by averaging predictions from multiple models.
8.  **Dropout**: In neural networks, randomly dropping out neurons during training.

### How to Fix Underfitting:
1.  **More Complex Model**: Use a model with more parameters or greater capacity (e.g., switch from Linear Regression to Polynomial Regression, or from Logistic Regression to Random Forest/Gradient Boosting).
2.  **Feature Engineering**: Create new features that capture more information or non-linear relationships.
3.  **Reduce Regularization**: If regularization was applied, reduce its strength to allow the model more flexibility.
4.  **Increase Training Time/Iterations**: For iterative models, train for longer.
5.  **Remove Noise**: Clean the data more thoroughly if there's significant noise hindering the model's ability to learn.

### Checking for Overfitting/Underfitting in our Models

We can examine the training and test set scores (e.g., accuracy, F1-score) for our trained models.


In [ ]:
if 'X_train_final' in locals() and 'y_train_final' in locals() and 'X_test_scaled' in locals() and 'y_test' in locals() and trained_models:
    logging.info("\n--- Checking for Overfitting/Underfitting ---")
    for name, model in trained_models.items():
        try:
            y_train_pred = model.predict(X_train_final)
            y_test_pred = model.predict(X_test_scaled)

            train_f1 = f1_score(y_train_final, y_train_pred, zero_division=0)
            test_f1 = f1_score(y_test, y_test_pred, zero_division=0)

            logging.info(f"{name}:")
            logging.info(f"  Training F1-Score: {train_f1:.4f}")
            logging.info(f"  Test F1-Score: {test_f1:.4f}")

            if train_f1 > test_f1 + 0.15: # Arbitrary threshold for significant difference
                logging.warning(f"  Potential Overfitting detected for {name}. Train F1 is significantly higher than Test F1.")
            elif train_f1 < 0.50 and test_f1 < 0.50: # Arbitrary threshold for low performance
                logging.warning(f"  Potential Underfitting detected for {name}. Both Train and Test F1 scores are low.")
            else:
                logging.info(f"  Model {name} shows reasonable balance between training and test performance.")
        except Exception as e:
            logging.error(f"Error checking overfitting/underfitting for {name}: {e}")
else:
    logging.error("Training or test data not available to check for overfitting/underfitting.")


Based on the F1-scores, we can observe the performance gap between training and test sets for each model.
*   A large gap (e.g., training F1 much higher than test F1) suggests overfitting.
*   Low F1 scores on both train and test suggest underfitting.
*   Models like Random Forest and Gradient Boosting, being more complex, have a higher potential for overfitting if not tuned properly or if the data is noisy. Logistic Regression, being simpler, is less likely to overfit but might underfit if the relationships are highly non-linear.


## 14. Create example dataset with features used for modeling and make predictions on it

To demonstrate how our model would work on new, unseen data, we'll create a small synthetic dataset with the same features used for training. We'll then apply the same preprocessing steps (scaling) and make predictions using our best-performing model.


In [ ]:
if 'scaler' in locals() and 'X' in locals() and trained_models:
    # Identify the best model based on F1-score from previous evaluation
    best_model_name = max(results, key=lambda k: results[k]['F1-Score'])
    best_model = trained_models[best_model_name]
    logging.info(f"Selected '{best_model_name}' as the best model for demonstration.")

    # Create a sample dataset with realistic values
    # Ensure all original feature names are present and in the correct order
    # from the original X (before scaling)
    sample_data = pd.DataFrame([
        {'male': 1, 'age': 55, 'education': 2.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0,
         'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 240.0, 'sysBP': 145.0,
         'diaBP': 90.0, 'BMI': 28.0, 'heartRate': 75.0, 'glucose': 95.0},
        {'male': 0, 'age': 40, 'education': 4.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0,
         'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 180.0, 'sysBP': 110.0,
         'diaBP': 70.0, 'BMI': 22.0, 'heartRate': 68.0, 'glucose': 80.0},
        {'male': 1, 'age': 68, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 1.0,
         'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 1, 'totChol': 280.0, 'sysBP': 160.0,
         'diaBP': 100.0, 'BMI': 32.0, 'heartRate': 90.0, 'glucose': 120.0}
    ], columns=X.columns) # Use original columns to maintain order

    logging.info("Created a sample dataset for prediction:")
    print(sample_data)

    try:
        # Scale the numerical features of the sample data using the *fitted* scaler
        sample_data_scaled = sample_data.copy()
        sample_data_scaled[numerical_features] = scaler.transform(sample_data[numerical_features])
        logging.info("Sample data scaled.")

        # Make predictions
        sample_predictions = best_model.predict(sample_data_scaled)
        sample_probabilities = best_model.predict_proba(sample_data_scaled)[:, 1] if hasattr(best_model, 'predict_proba') else ['N/A'] * len(sample_data_scaled)

        logging.info("\nPredictions on the sample dataset:")
        for i, (pred, prob) in enumerate(zip(sample_predictions, sample_probabilities)):
            prediction_text = "Positive (CHD risk)" if pred == 1 else "Negative (No CHD risk)"
            prob_text = f" (Probability: {prob:.4f})" if isinstance(prob, float) else ""
            logging.info(f"Sample {i+1}: Predicted class: {prediction_text}{prob_text}")
    except Exception as e:
        logging.error(f"Error making predictions on sample data: {e}")
else:
    logging.error("Scaler or best model not available for new data prediction.")


## 15. Hyperparameter tuning on sample or small dataset

Hyperparameter tuning is the process of finding the optimal set of hyperparameters for a machine learning model to maximize its performance. Instead of using default values, we systematically search through a predefined range of values.

For demonstration, we will perform hyperparameter tuning using `GridSearchCV` on a **small subset of the training data** to keep the computation time reasonable as per the instructions. In a real-world scenario, this would be done on the full training set using cross-validation. We will tune the `RandomForestClassifier`, as it's a powerful model with several important hyperparameters.

**Hyperparameters to tune for RandomForestClassifier:**
*   `n_estimators`: Number of trees in the forest.
*   `max_features`: The number of features to consider when looking for the best split.
*   `max_depth`: The maximum depth of the tree.
*   `min_samples_split`: The minimum number of samples required to split an internal node.
*   `min_samples_leaf`: The minimum number of samples required to be at a leaf node.


In [ ]:
if 'X_train_final' in locals() and 'y_train_final' in locals():
    logging.info("\n--- Starting Hyperparameter Tuning for Random Forest Classifier ---")

    # Use a smaller subset of the training data for faster tuning as per instruction
    # For a full run, comment out this subsampling
    X_train_tuning, _, y_train_tuning, _ = train_test_split(X_train_final, y_train_final, test_size=0.8, random_state=42, stratify=y_train_final)
    logging.info(f"Using a subset of training data for tuning: {X_train_tuning.shape}")

    # Define the parameter grid
    param_grid = {
        'n_estimators': [50, 100], # Reduced for speed
        'max_features': ['sqrt', None], # 'auto' is now 'sqrt'
        'max_depth': [5, 10], # Reduced for speed
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2]
    }

    rf_model_tuning = RandomForestClassifier(random_state=42)

    try:
        # Initialize GridSearchCV
        grid_search = GridSearchCV(estimator=rf_model_tuning, param_grid=param_grid,
                                   cv=3, n_jobs=-1, verbose=1, scoring='f1', error_score='raise') # cv=3 for speed

        # Fit GridSearchCV
        grid_search.fit(X_train_tuning, y_train_tuning)

        logging.info("Hyperparameter tuning complete.")
        logging.info(f"Best parameters found: {grid_search.best_params_}")
        logging.info(f"Best F1-score (cross-validated): {grid_search.best_score_:.4f}")

        # Update the best model with tuned parameters
        tuned_rf_model = grid_search.best_estimator_
        trained_models['Tuned Random Forest'] = tuned_rf_model
        logging.info("Tuned Random Forest Classifier added to trained models.")

        # Re-evaluate all models including the tuned one to see improvement
        logging.info("\nRe-evaluating models including the tuned Random Forest...")
        results = {}
        roc_curves = {}
        for name, model in trained_models.items():
            try:
                y_pred = model.predict(X_test_scaled)
                y_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, 'predict_proba') else [0] * len(y_test)

                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred, zero_division=0)
                recall = recall_score(y_test, y_pred, zero_division=0)
                f1 = f1_score(y_test, y_pred, zero_division=0)
                roc_auc = roc_auc_score(y_test, y_proba) if hasattr(model, 'predict_proba') else 0

                results[name] = {
                    'Accuracy': accuracy,
                    'Precision': precision,
                    'Recall': recall,
                    'F1-Score': f1,
                    'ROC AUC': roc_auc
                }
                if hasattr(model, 'predict_proba'):
                    fpr, tpr, _ = roc_curve(y_test, y_proba)
                    roc_curves[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}

            except Exception as e:
                logging.error(f"Error during re-evaluation of {name} after tuning: {e}")

        results_df_tuned = pd.DataFrame(results).T
        print("\n--- Model Evaluation Results (After Tuning) ---")
        print(results_df_tuned.sort_values(by='F1-Score', ascending=False))

    except Exception as e:
        logging.error(f"Error during hyperparameter tuning: {e}")
else:
    logging.error("Training data not available for hyperparameter tuning.")


## 16. Visual representation of the results, explain the comparison between predicted and true data.

After training and potentially tuning our models, visualizing the results is critical for understanding their performance. We will focus on:
1.  **Confusion Matrices**: To clearly see where each model is making errors (FP, FN).
2.  **ROC Curves**: To compare the overall discriminative power of the models, especially useful for imbalanced datasets.
3.  **Predicted vs. True Labels (for a subset)**: To visually inspect how closely predicted probabilities align with actual outcomes.


In [ ]:
if 'X_test_scaled' in locals() and 'y_test' in locals() and trained_models:
    logging.info("\n--- Visualizing Final Model Results ---")

    # Plot Confusion Matrices for all models (including tuned RF if available)
    num_models = len(trained_models)
    rows = (num_models + 1) // 2
    cols = 2 if num_models > 0 else 1

    fig_cm = make_subplots(rows=rows, cols=cols, subplot_titles=[f'Confusion Matrix: {name}' for name in trained_models.keys()])

    for i, (name, model) in enumerate(trained_models.items()):
        row = (i // cols) + 1
        col = (i % cols) + 1
        try:
            y_pred = model.predict(X_test_scaled)
            cm = confusion_matrix(y_test, y_pred)
            
            z = [[cm[0,0], cm[0,1]], [cm[1,0], cm[1,1]]]
            x = ['Predicted 0', 'Predicted 1']
            y = ['Actual 0', 'Actual 1']
            
            heatmap_trace = go.Heatmap(z=z, x=x, y=y, colorscale='Blues',
                                       text=[[str(val) for val in row_cm] for row_cm in cm],
                                       texttemplate="%{text}", textfont={"size":15},
                                       showscale=False) # Hide color scale for individual plots
            fig_cm.add_trace(heatmap_trace, row=row, col=col)
            fig_cm.update_xaxes(title_text="Predicted Class", row=row, col=col)
            fig_cm.update_yaxes(title_text="Actual Class", row=row, col=col)

        except Exception as e:
            logging.error(f"Error generating confusion matrix for {name}: {e}")

    fig_cm.update_layout(height=400 * rows, width=400 * cols, title_text='Confusion Matrices for All Models')
    fig_cm.show()
    logging.info("Displayed confusion matrices.")

    # Plot ROC curves for all models
    fig_roc = go.Figure()
    fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier (AUC = 0.50)',
                                 line=dict(dash='dash', color='gray')))

    roc_curves_final = {}
    for name, model in trained_models.items():
        try:
            if hasattr(model, 'predict_proba'):
                y_proba = model.predict_proba(X_test_scaled)[:, 1]
                fpr, tpr, _ = roc_curve(y_test, y_proba)
                roc_auc = auc(fpr, tpr)
                roc_curves_final[name] = {'fpr': fpr, 'tpr': tpr, 'auc': roc_auc}
                fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                                             name=f'{name} (AUC = {roc_auc:.2f})'))
        except Exception as e:
            logging.error(f"Error generating ROC curve for {name}: {e}")

    fig_roc.update_layout(title='ROC Curve Comparison (All Models)',
                          xaxis_title='False Positive Rate',
                          yaxis_title='True Positive Rate',
                          xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1]),
                          showlegend=True)
    fig_roc.show()
    logging.info("Displayed ROC curve comparison.")

    # Visualize predicted probabilities vs true labels for a subset
    # Let's take the best model (after tuning)
    best_model_name_tuned = max(results_df_tuned.index, key=lambda k: results_df_tuned.loc[k, 'F1-Score'])
    final_best_model = trained_models[best_model_name_tuned]

    if hasattr(final_best_model, 'predict_proba'):
        y_test_proba = final_best_model.predict_proba(X_test_scaled)[:, 1]
        
        # Create a DataFrame for easy plotting
        plot_df = pd.DataFrame({'Actual': y_test, 'Predicted_Proba': y_test_proba}).sample(n=min(500, len(y_test)), random_state=42) # Sample for readability

        fig_proba = px.scatter(plot_df, x='Predicted_Proba', y='Actual', color='Actual',
                               title=f'Predicted Probabilities vs. Actual Labels ({best_model_name_tuned})',
                               labels={'Predicted_Proba': 'Predicted Probability of CHD', 'Actual': 'Actual CHD (0/1)'},
                               color_continuous_scale='Viridis',
                               range_x=[0,1], range_y=[-0.1, 1.1])
        fig_proba.update_traces(marker=dict(size=8, opacity=0.7), selector=dict(mode='markers'))
        fig_proba.add_trace(go.Scatter(x=[0.5, 0.5], y=[0, 1], mode='lines', name='Decision Threshold (0.5)',
                                       line=dict(dash='dash', color='red')))
        fig_proba.show()
        logging.info(f"Displayed predicted probabilities vs. actual labels for {best_model_name_tuned}.")
    else:
        logging.warning("Selected best model does not have 'predict_proba' method for probability visualization.")

else:
    logging.error("Test data or trained models not available for results visualization.")


### Explanation of Results Visualizations

**Confusion Matrices:**
These plots provide a granular view of how each model performs.
*   The top-left cell (True Negative) shows how many healthy individuals were correctly identified.
*   The top-right cell (False Positive) shows how many healthy individuals were incorrectly classified as having CHD.
*   The bottom-left cell (False Negative) shows how many individuals with CHD were incorrectly classified as healthy. This is a critical error in medical contexts.
*   The bottom-right cell (True Positive) shows how many individuals with CHD were correctly identified.

By comparing the confusion matrices across models, we can see which model has a better balance of avoiding FPs and FNs, depending on the specific cost associated with each type of error. For CHD prediction, minimizing False Negatives (missing a CHD patient) is often paramount.

**ROC Curve Comparison:**
The ROC (Receiver Operating Characteristic) curve plots the True Positive Rate (Recall) against the False Positive Rate at various classification thresholds.
*   The area under the ROC curve (AUC) quantifies the overall performance. An AUC of 1.0 means perfect classification, while 0.5 means random guessing (like the dashed gray line).
*   Models with curves closer to the top-left corner and higher AUC values are better at distinguishing between the two classes.
*   This plot is particularly useful for imbalanced datasets as it evaluates the model's performance across all possible thresholds, without being biased by class distribution.

**Predicted Probabilities vs. Actual Labels:**
This scatter plot (for a subset of data) shows the predicted probability of `TenYearCHD=1` against the actual `TenYearCHD` label.
*   Ideally, all points for `Actual=0` would have predicted probabilities close to 0, and all points for `Actual=1` would have predicted probabilities close to 1.
*   The red dashed line represents the default classification threshold (0.5).
*   Points where `Actual=0` but `Predicted_Proba` is above 0.5 are False Positives.
*   Points where `Actual=1` but `Predicted_Proba` is below 0.5 are False Negatives.
This plot helps to visually assess the model's calibration and where its confidence aligns or diverges from reality.


## 17. Final model selection based on best result.

Based on our evaluation metrics, especially F1-Score and ROC AUC (given the class imbalance), we select the model that demonstrated the best overall performance. The F1-Score provides a balance between Precision and Recall, which is crucial when both types of errors (false positives and false negatives) have significant consequences. ROC AUC indicates the model's ability to discriminate between positive and negative classes across various thresholds.


In [ ]:
if 'results_df_tuned' in locals():
    # Find the model with the highest F1-Score
    final_best_model_name = results_df_tuned['F1-Score'].idxmax()
    final_best_model = trained_models[final_best_model_name]

    logging.info(f"\n--- Final Model Selection ---")
    logging.info(f"The best performing model based on F1-Score is: '{final_best_model_name}'")
    logging.info(f"Its performance metrics are:\n{results_df_tuned.loc[final_best_model_name]}")

    selected_model_for_saving = final_best_model
else:
    logging.error("Model evaluation results not available for final model selection.")
    selected_model_for_saving = None # Fallback if results are missing


## 18. Ensure to save the final model using pickle library and create a folder named artifacts to store the model.

It's crucial to save the trained machine learning model so it can be re-used later for predictions without needing to retrain it. We will use the `pickle` library for this purpose and store the model in an `artifacts` folder created at the root directory.


In [ ]:
if selected_model_for_saving is not None:
    # Define the path to save the model
    model_filename = f'{final_best_model_name.replace(" ", "_").lower()}_chd_model.pkl'
    model_filepath = os.path.join(ARTIFACTS_DIR, model_filename)

    try:
        with open(model_filepath, 'wb') as file:
            pickle.dump(selected_model_for_saving, file)
        logging.info(f"Final model '{final_best_model_name}' successfully saved to '{model_filepath}'")
    except Exception as e:
        logging.error(f"Error saving the model: {e}")
else:
    logging.error("No model selected for saving. Skipping model saving step.")

# Example of how to load the model (for future use)
# try:
#     with open(model_filepath, 'rb') as file:
#         loaded_model = pickle.load(file)
#     logging.info(f"Model loaded successfully from '{model_filepath}' for verification.")
#     # You can then use loaded_model.predict()
# except Exception as e:
#     logging.error(f"Error loading the saved model: {e}")


## 19. Insights

Based on our comprehensive analysis and modeling, here are some key insights regarding the prediction of 10-year Coronary Heart Disease risk:

1.  **Key Risk Factors**: The EDA and correlation analysis confirmed that `age`, `sysBP`, `glucose`, `totChol`, `prevalentHyp`, and `diabetes` are significant positive indicators for CHD risk. `male` gender also appears as a consistent risk factor. These are well-aligned with medical understanding of cardiovascular disease.
2.  **Class Imbalance**: The target variable `TenYearCHD` is highly imbalanced, with a much smaller number of positive cases (individuals developing CHD). This necessitated the use of techniques like `stratify` during train-test split and `SMOTE` for oversampling the minority class in the training data. Without these, models would likely be biased towards predicting the majority class (no CHD).
3.  **Model Performance**: Ensemble models like Random Forest and Gradient Boosting generally performed better than Logistic Regression and SVC, especially in terms of F1-score and ROC AUC. This suggests that the relationship between risk factors and CHD is likely non-linear and complex, which these models are better equipped to capture.
4.  **Hyperparameter Tuning Impact**: Tuning the hyperparameters of the Random Forest Classifier led to an improvement in performance, demonstrating the importance of optimizing model parameters for the specific dataset.
5.  **Importance of Metrics**: For this problem, F1-score and ROC AUC are more informative than simple accuracy due to the imbalanced nature of the target variable. Minimizing False Negatives (missing actual CHD cases) is often a critical objective in medical diagnosis, making Recall and F1-score particularly important.
6.  **Outlier Handling**: Capping outliers in numerical features helped in creating a more robust dataset, potentially preventing extreme values from skewing model training.
7.  **Data Quality**: The dataset required imputation for missing values, highlighting the common challenge of incomplete data in real-world scenarios.

These insights can guide public health interventions, personalized risk assessments, and further research into CHD prevention.


## 20. Conclusion

This Jupyter notebook successfully outlines and executes a machine learning project for predicting the 10-year risk of Coronary Heart Disease. We started with data loading and moved through detailed EDA, robust preprocessing (including handling missing values and outliers, and addressing class imbalance), and comprehensive model training.

We evaluated several classification models using appropriate metrics for an imbalanced dataset, such as F1-score and ROC AUC, and demonstrated the process of hyperparameter tuning to optimize model performance. Concepts like gradient descent, residuals, and overfitting/underfitting were explained and illustrated to provide a deeper understanding of the machine learning process.

The "Tuned Random Forest Classifier" emerged as the best-performing model based on our evaluation metrics. This final model was saved for future deployment, ensuring that our efforts can be readily utilized for real-world predictions.

**Future Work and Improvements:**
*   **More Advanced Feature Engineering**: Explore creating more sophisticated interaction terms or polynomial features based on domain expertise.
*   **Alternative Imputation Strategies**: Investigate more advanced imputation techniques like MICE (Multiple Imputation by Chained Equations).
*   **Deep Learning Models**: For very large datasets, neural networks could be explored, although for tabular data, ensemble methods often perform comparably or better.
*   **Explainable AI (XAI)**: Implement techniques like SHAP or LIME to further understand individual feature contributions and model decisions, which is crucial in medical contexts.
*   **Cost-Sensitive Learning**: Integrate cost matrices directly into model training to explicitly prioritize minimizing False Negatives over False Positives, given the potential severity of missing a CHD diagnosis.
*   **External Validation**: Test the model on an entirely new, independent dataset to ensure true generalization.

This project provides a strong foundation for CHD risk prediction, offering a valuable tool for healthcare professionals and researchers.

---
End of Notebook
---